# A.X-Encoder-base 문맥 적절성 평가 (career-jikimi)

`skt/A.X-Encoder-base`를 100행 데이터로 평가한다.

## 시작 전에 알아야 할 것 두 가지

**1. 이 모델은 zero-shot 평가가 불가능하다.**
A.X-Encoder는 사전학습된 인코더일 뿐 분류 헤드가 없다. `num_labels=2`로 불러오면
헤드가 **무작위로 초기화**되므로, 학습 없이 예측하면 동전 던지기(≈0.5)가 나온다.
따라서 "평가"는 반드시 **파인튜닝 + 교차검증** 형태여야 한다. 이 노트북이 그렇게 한다.

**2. 이 100행에는 심각한 라벨 누출이 있다.**
사전 진단 결과 **response(응답 문장)만 보고 정확도 0.940**이 나온다.
고유 response 100개가 전부 한쪽 라벨에만 등장하기 때문이다 — 부적절은 전부 잡담체,
적절은 전부 화제 관련. 즉 **history를 아예 안 봐도 94%를 맞출 수 있다.**

> 그래서 이 노트북의 결론은 "정확도 몇 %"가 아니라
> **"그 정확도가 문맥에서 온 것인가, 응답 표면에서 온 것인가"** 다.
> 이걸 §5 절제 실험이 판정한다.

## 실행 순서

1. 설치 · 데이터 로드
2. **누출 진단** (파인튜닝 전에 먼저 — 기준선을 알아야 결과를 읽을 수 있다)
3. 모델 로드 · 환경 점검
4. GroupKFold 교차검증 파인튜닝
5. **history 절제 실험** ← 이 노트북의 핵심
6. 결과 종합 · 오분류 분석

**런타임 → 런타임 유형 변경 → GPU (T4면 충분)** 로 설정하고 시작한다.
전체 5~10분.


## 1. 설치 · 데이터 로드

In [ ]:
# ModernBERT 아키텍처 지원은 transformers 4.48부터다. 이 버전 미만이면 A.X-Encoder가 로드되지 않는다.
!pip -q install -U "transformers>=4.48" "scikit-learn>=1.3" pandas
import transformers, torch
print("transformers", transformers.__version__)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[!] GPU가 없다. 런타임 → 런타임 유형 변경 → T4 GPU 로 바꾸고 다시 실행할 것.")

In [ ]:
CSV_PATH = "dialogue_context_appropriateness_100.csv"   # 다른 파일로 평가하려면 여기만 바꾼다

import os, pandas as pd
if not os.path.exists(CSV_PATH):
    from google.colab import files
    up = files.upload()
    CSV_PATH = list(up.keys())[0]

df = pd.read_csv(CSV_PATH)
print(CSV_PATH, "|", df.shape)

# 스키마 확인 — 아래 4개 컬럼만 사용한다
need = {"pair_id", "history", "response", "label"}
missing = need - set(df.columns)
assert not missing, f"컬럼 누락: {missing} (있는 컬럼: {list(df.columns)})"

df = df.dropna(subset=["history", "response", "label"]).reset_index(drop=True)
df["history"] = df["history"].astype(str)
df["response"] = df["response"].astype(str)

print("\n라벨 분포:", df["label"].value_counts().to_dict())
print("고유 response:", df["response"].nunique(), "| 고유 history:", df["history"].nunique(),
      "| pair:", df["pair_id"].nunique())
df.head(4)

In [ ]:
import numpy as np

# 부적절 = 1 (양성). 프로젝트의 우선 지표가 "부적절 판정의 precision"이므로 부적절을 양성으로 둔다.
LABEL_POS = "부적절"
y = (df["label"] == LABEL_POS).astype(int).values
groups = df["pair_id"].astype(str).values

assert set(np.unique(y)) == {0, 1}, f"라벨 값이 이상하다: {df['label'].unique()}"
print(f"양성(={LABEL_POS}) {y.sum()}행 / 음성 {(1-y).sum()}행 / 그룹(pair) {len(set(groups))}개")

# 같은 pair가 학습·검증에 쪼개져 들어가면 검증이 무의미해진다 (쌍둥이 행이 정답을 흘린다).
# 그래서 아래 모든 분할은 pair_id 그룹 단위로 한다.

## 2. 누출 진단 — 파인튜닝 **전에** 한다

순서가 중요하다. A.X-Encoder 점수를 먼저 보면 0.9x라는 숫자에 anchor가 걸려서
"잘 나왔다"고 읽게 된다. 기준선을 먼저 박아두면 같은 숫자가 다르게 읽힌다.

세 가지를 잰다. 전부 문자 n-gram TF-IDF + 로지스틱 회귀라는 **아주 단순한** 모델이다.
단순한 모델이 잘 맞힌다는 건 곧 정답이 표면에 있다는 뜻이다.

| 진단 | 입력 | 이 값이 높으면 |
|---|---|---|
| response-only | 응답만 | **라벨이 응답 표면에 있다 = 누출** |
| history-only | 대화만 | 방 자체가 라벨을 결정한다 = 누출 |
| history+response | 둘 다 | 참고용 |


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score

def surface_baseline(texts, name):
    pipe = make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1),
        LogisticRegression(max_iter=2000),
    )
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    pred = cross_val_predict(pipe, np.array(texts), y, cv=cv, groups=groups)
    acc = accuracy_score(y, pred)
    print(f"{name:22s} acc = {acc:.3f}")
    return acc

print("=== 표면 특징 기준선 (5-fold, pair 그룹 분할) ===")
BASE_RESP = surface_baseline(df["response"], "response-only")
BASE_HIST = surface_baseline(df["history"], "history-only")
BASE_BOTH = surface_baseline(df["history"] + " [SEP] " + df["response"], "history+response")

# 같은 response가 양쪽 라벨에 등장하는가? 0이면 response와 라벨이 1:1로 묶여 있다는 뜻.
both_sides = (df.groupby("response")["label"].nunique() > 1).sum()
print(f"\n양쪽 라벨에 모두 등장하는 response: {both_sides} / {df['response'].nunique()}")

print("\n" + "=" * 58)
if BASE_RESP > 0.65:
    print(f"[누출] response만으로 {BASE_RESP:.3f}. 이 데이터의 정확도는")
    print("       문맥 이해가 아니라 응답 문체를 잰 값일 수 있다.")
    print("       → §5 절제 실험을 반드시 확인할 것.")
else:
    print(f"[정상] response-only {BASE_RESP:.3f} — 표면 누출이 통제된 데이터다.")
print("=" * 58)

## 3. 모델 로드 · 환경 점검

`AutoModelForSequenceClassification`으로 불러오면
`Some weights ... newly initialized: ['classifier.weight', ...]` 경고가 뜬다.

**이건 오류가 아니라 정상이다.** 분류 헤드가 새로 생긴다는 뜻이고,
동시에 이 모델을 학습 없이 평가할 수 없는 이유이기도 하다.


In [ ]:
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "skt/A.X-Encoder-base"
MAX_LEN    = 256     # history 3턴 + response는 100토큰 내외. 긴 문맥 데이터셋을 쓰면 512로 올린다.
EPOCHS     = 6
BATCH      = 8
LR         = 2e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# T4(Turing)는 bf16을 지원하지 않는다. Ampere(8.x) 이상에서만 bf16을 쓴다.
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
USE_BF16 = cap[0] >= 8
ATTN = "sdpa" if torch.cuda.is_available() else "eager"
print(f"device={device} | compute capability={cap} | bf16={USE_BF16} | attn={ATTN}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    """fold마다 새 모델을 만든다 (가중치가 fold 간에 새면 CV가 무의미해진다)."""
    try:
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2, attn_implementation=ATTN)
    except Exception as e:
        print(f"[fallback] attn={ATTN} 실패 → eager 로 재시도 ({type(e).__name__})")
        m = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=2, attn_implementation="eager")
    return m.to(device)

_m = load_model()
n_param = sum(p.numel() for p in _m.parameters())
print(f"\n파라미터 {n_param/1e6:.1f}M | 최대 컨텍스트 {tokenizer.model_max_length}")
del _m; torch.cuda.empty_cache()

In [ ]:
def encode(histories, responses):
    """cross-encoder 입력: [CLS] history [SEP] response [SEP]
    잘릴 때는 history 쪽만 자른다(only_first) — 판정 대상인 response는 온전해야 한다."""
    return tokenizer(
        list(histories), list(responses),
        truncation="only_first", max_length=MAX_LEN,
        padding=True, return_tensors="pt",
    )

# 토큰 길이 분포 — MAX_LEN이 충분한지 확인
lens = [len(tokenizer(h, r)["input_ids"]) for h, r in zip(df["history"], df["response"])]
print(f"토큰 길이  중앙값 {int(np.median(lens))} | p95 {int(np.percentile(lens, 95))} | 최대 {max(lens)}")
if max(lens) > MAX_LEN:
    print(f"[!] {sum(l > MAX_LEN for l in lens)}행이 잘린다. MAX_LEN을 올릴 것.")
else:
    print(f"MAX_LEN={MAX_LEN} 로 전 행이 잘리지 않는다.")

## 4. 교차검증 파인튜닝

**5-fold × 3 seed = 15회 학습.** 100행에서 1회 CV만 하면 fold 운에 따라 정확도가
±10%p씩 흔들린다. seed를 바꿔 3번 반복해 **표준편차까지** 낸다 —
단일 숫자보다 "0.93 ± 0.04"가 정직하다.

분할은 **pair_id 그룹 단위**다. 같은 history를 공유하는 적절/부적절 쌍이
학습과 검증에 갈라져 들어가면 검증 점수가 부풀려진다.

각 fold에서 검증 세트를 **세 가지 조건**으로 예측한다 (§5에서 쓴다):

- `normal` — 원래 history
- `swapped` — 다른 방의 history로 교체
- `empty` — history 없음


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from transformers import get_linear_schedule_with_warmup

def train_one_fold(tr_idx, va_idx, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = load_model()

    enc = encode(df["history"].values[tr_idx], df["response"].values[tr_idx])
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"],
                       torch.tensor(y[tr_idx], dtype=torch.long))
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=False)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(total * 0.1), total)
    amp = torch.bfloat16 if USE_BF16 else torch.float32

    model.train()
    for _ in range(EPOCHS):
        for ids, mask, lab in dl:
            ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                loss = model(input_ids=ids, attention_mask=mask, labels=lab).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()

    @torch.no_grad()
    def predict(hists, resps):
        model.eval()
        out = []
        for i in range(0, len(resps), 32):
            e = encode(hists[i:i+32], resps[i:i+32])
            with torch.autocast("cuda", dtype=amp, enabled=USE_BF16):
                lg = model(input_ids=e["input_ids"].to(device),
                           attention_mask=e["attention_mask"].to(device)).logits
            out.append(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
        return np.concatenate(out)

    h_va, r_va = df["history"].values[va_idx], df["response"].values[va_idx]
    # swapped: 검증 세트 안에서 history를 한 칸씩 밀어 다른 방의 문맥과 짝지운다
    h_swap = np.roll(h_va, 1) if len(h_va) > 1 else h_va
    res = {
        "normal":  predict(h_va, r_va),
        "swapped": predict(h_swap, r_va),
        "empty":   predict(np.array([""] * len(r_va)), r_va),
    }
    del model; torch.cuda.empty_cache()
    return res

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import time

SEEDS = [42, 7, 2026]
CONDS = ["normal", "swapped", "empty"]
oof = {s: {c: np.zeros(len(df)) for c in CONDS} for s in SEEDS}

t0 = time.time()
for seed in SEEDS:
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    for k, (tr, va) in enumerate(cv.split(df, y, groups)):
        r = train_one_fold(tr, va, seed)
        for c in CONDS:
            oof[seed][c][va] = r[c]
        print(f"  seed {seed} fold {k+1}/5 완료 ({time.time()-t0:.0f}s)", end="\r")
    acc = ((oof[seed]["normal"] > 0.5).astype(int) == y).mean()
    print(f"seed {seed}: OOF acc = {acc:.3f}" + " " * 20)

print(f"\n총 {time.time()-t0:.0f}초")

## 5. history 절제 실험 — 이 노트북의 핵심

학습된 모델에게 **응답은 그대로 두고 history만 망가뜨려** 예측을 다시 받는다.

| 조건 | 해석 |
|---|---|
| `normal` | 원래 성능 |
| `swapped` | 다른 방 문맥. **여기서 안 떨어지면 모델이 history를 안 본다** |
| `empty` | 문맥 없음. 순수 응답 표면 성능 |

논리는 단순하다 — 문맥 부적합을 판정하는 모델이라면 문맥을 바꿨을 때
**같은 응답의 판정이 뒤집혀야** 한다. 안 뒤집히면 문맥을 읽는 게 아니다.

특히 `swapped`는 원래 적절했던 응답을 부적절하게 만드므로,
**정확도가 0.5 아래로 떨어지는 것이 정상**이다. 0.9 근처를 유지하면 그게 이상한 것이다.


In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for c in CONDS:
    accs = [(((oof[s][c] > 0.5).astype(int)) == y).mean() for s in SEEDS]
    aucs = [roc_auc_score(y, oof[s][c]) for s in SEEDS]
    rows.append({"조건": c,
                 "정확도": f"{np.mean(accs):.3f} ± {np.std(accs):.3f}",
                 "AUC":   f"{np.mean(aucs):.3f} ± {np.std(aucs):.3f}",
                 "_acc": np.mean(accs)})
abl = pd.DataFrame(rows)
print(abl[["조건", "정확도", "AUC"]].to_string(index=False))

a_norm, a_swap, a_empty = abl["_acc"].values
drop_swap  = a_norm - a_swap
drop_empty = a_norm - a_empty

print(f"\nswapped 하락폭 : {drop_swap:+.3f}")
print(f"empty   하락폭 : {drop_empty:+.3f}")
print("=" * 62)
if drop_swap < 0.10:
    print("[문맥 미사용] history를 다른 방 것으로 바꿔도 성능이 유지된다.")
    print("  → 모델은 응답 문체만 보고 있다. 정확도가 아무리 높아도")
    print("     '문맥 부적합 판정'의 근거로 쓸 수 없다.")
    print("  → 원인은 모델이 아니라 데이터다. 같은 response가 문맥에 따라")
    print("     양쪽 라벨을 갖는 짝 구조(v2.5/v3 방식)로 재조립해야 한다.")
elif drop_swap < 0.30:
    print("[부분 사용] 문맥을 보긴 하지만 응답 표면 의존이 남아 있다.")
else:
    print("[문맥 사용] 문맥을 바꾸면 판정이 뒤집힌다. 의도한 대로 동작한다.")
print("=" * 62)

## 6. 성능 지표 · 기준선 대조

In [ ]:
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

p_mean = np.mean([oof[s]["normal"] for s in SEEDS], axis=0)
pred = (p_mean > 0.5).astype(int)

print("=== A.X-Encoder-base · OOF (seed 3회 평균) ===")
print(classification_report(y, pred, target_names=["적절", "부적절"], digits=3))

cm = confusion_matrix(y, pred)
print("혼동행렬            예측:적절  예측:부적절")
print(f"  실제 적절          {cm[0,0]:5d}     {cm[0,1]:5d}   <- 우상단 = false positive (멀쩡한 메시지에 팝업)")
print(f"  실제 부적절        {cm[1,0]:5d}     {cm[1,1]:5d}")

print("\n=== 기준선 대조 ===")
comp = pd.DataFrame([
    {"방법": "response-only (TF-IDF)", "정확도": f"{BASE_RESP:.3f}", "비고": "응답 표면만"},
    {"방법": "history-only  (TF-IDF)", "정확도": f"{BASE_HIST:.3f}", "비고": "문맥 표면만"},
    {"방법": "A.X-Encoder (empty)",    "정확도": f"{a_empty:.3f}",   "비고": "문맥 제거"},
    {"방법": "A.X-Encoder (normal)",   "정확도": f"{a_norm:.3f}",    "비고": "전체"},
])
print(comp.to_string(index=False))

gain = a_norm - BASE_RESP
print(f"\nA.X-Encoder(normal) - response-only 기준선 = {gain:+.3f}")
if gain < 0.03:
    print("  → 0.1B 인코더를 파인튜닝한 값어치가 TF-IDF 대비 거의 없다.")
    print("     모델 문제가 아니라 데이터가 문맥을 요구하지 않는다는 뜻이다.")

In [ ]:
# threshold 스윕 — 프로젝트 우선 지표는 precision이다 (false positive가 치명적)
print("thr   precision  recall     F1     팝업률")
best = None
for thr in np.arange(0.05, 1.00, 0.05):
    pr = (p_mean > thr).astype(int)
    if pr.sum() == 0:
        continue
    p = precision_score(y, pr, zero_division=0)
    r = recall_score(y, pr, zero_division=0)
    f = f1_score(y, pr, zero_division=0)
    print(f"{thr:.2f}    {p:.3f}     {r:.3f}   {f:.3f}    {pr.mean():.3f}")
    if best is None or f > best[3]:
        best = (thr, p, r, f)
print(f"\nF1 최대: thr={best[0]:.2f} (precision {best[1]:.3f} / recall {best[2]:.3f})")

ok = [(t, precision_score(y, (p_mean > t).astype(int), zero_division=0),
          recall_score(y, (p_mean > t).astype(int), zero_division=0))
      for t in np.arange(0.05, 1.00, 0.05)
      if (p_mean > t).sum() > 0 and precision_score(y, (p_mean > t).astype(int), zero_division=0) >= 0.95]
if ok:
    t, p, r = ok[0]
    print(f"precision ≥ 0.95 를 만족하는 최소 thr = {t:.2f} (recall {r:.3f})")
else:
    print("precision 0.95 를 만족하는 threshold 없음")

In [ ]:
# 오분류 행 — 다음 데이터 증분의 근거가 여기서 나온다
df_err = df.copy()
df_err["prob_부적절"] = p_mean.round(3)
df_err["예측"] = np.where(pred == 1, "부적절", "적절")
df_err["정답여부"] = np.where(pred == y, "O", "X")

wrong = df_err[df_err["정답여부"] == "X"]
print(f"오분류 {len(wrong)} / {len(df)} 행\n")
for _, r in wrong.iterrows():
    print(f"[no={r['no'] if 'no' in r else '-'}] 정답={r['label']} 예측={r['예측']} (p={r['prob_부적절']})")
    print(f"  history : {r['history'][:70].replace(chr(10), ' / ')}...")
    print(f"  response: {r['response']}\n")

df_err.to_csv("oof_predictions_100.csv", index=False, encoding="utf-8-sig")
print("저장: oof_predictions_100.csv  (엑셀에서 바로 열림 — utf-8-sig)")

## 7. 결과 읽는 법

### 판정 순서

1. **§5 `swapped` 하락폭**을 먼저 본다. 0.10 미만이면 나머지 숫자는 의미가 없다 —
   모델이 문맥을 안 보고 있다는 뜻이고, 정확도는 응답 문체 암기 점수다.
2. 하락폭이 크면 그때 §6 precision을 본다. 프로젝트 우선 지표는 recall이 아니라
   **precision**이다 (멀쩡한 메시지에 팝업이 뜨면 아무도 안 쓴다).
3. `± 표준편차`를 항상 같이 읽는다. 100행 5-fold에서 fold당 검증은 20행뿐이라
   한 행이 5%p다. 소수점 셋째 자리를 신뢰하면 안 된다.

### 예상되는 결과와 대응

| 나온 결과 | 의미 | 다음 할 일 |
|---|---|---|
| normal 높고 swapped도 높음 | 데이터가 문맥을 요구하지 않음 | **데이터 재조립** (아래) |
| normal 높고 swapped 낮음 | 정상 동작 | 1,000행으로 확대, threshold 확정 |
| normal이 낮음 (0.7 미만) | 학습 부족 | `EPOCHS` 10, `LR` 3e-5 로 재시도 |

### 데이터를 고치는 방법은 이미 프로젝트 안에 있다

`dataset/` 의 v2.5·v3 작업이 정확히 이 문제를 푼 기록이다 —
**고유 response 전원이 적절 1회 + 부적절 1회로 등장하는 짝 구조**를 만들면
response-only 정확도가 정의상 0.500으로 떨어진다.
`training_dataset_v3_1000.csv`(1,000행)가 그 결과물이고, 검증 수치가
response-only 0.500 / 셔플 0.498 로 이미 통과돼 있다.

**§1의 `CSV_PATH`만 그 파일로 바꾸고 전부 재실행하면 같은 진단이 그대로 돈다.**
두 데이터셋의 `swapped` 하락폭을 나란히 놓는 것이 이 노트북의 가장 좋은 사용법이고,
발표에서 "데이터 설계가 왜 필요했는가"를 한 장으로 보여주는 근거가 된다.


In [ ]:
# 최종 모델을 전체 데이터로 학습해 저장하려면 아래 주석을 푼다.
# 단, §5에서 문맥 미사용으로 나왔다면 저장할 이유가 없다 — 데이터부터 고칠 것.
#
# torch.manual_seed(42)
# final = load_model()
# ... (train_one_fold 의 학습 루프를 전체 인덱스로 실행)
# final.save_pretrained("ax_encoder_final"); tokenizer.save_pretrained("ax_encoder_final")
# from google.colab import files; !zip -qr ax_encoder_final.zip ax_encoder_final
# files.download("ax_encoder_final.zip")
print("완료.")